# 03. CNNs and a Correct Training Baseline

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner → research-practical  
**Course:** Deep Learning for Optical Imaging

This notebook teaches the minimum end-to-end training pipeline that should work before you reproduce a complex paper.

> **How to study this notebook:** read the explanation first, predict what the code should do, run it, change one parameter, and explain why the result changed.


## Learning objectives

- Explain convolution and receptive fields
- Build a small CNN
- Perform a correct training step
- Separate training and validation behavior
- Define criteria for accepting a baseline


## Mind map

```mermaid
mindmap
  root((CNN baseline))
    Data
      Batch
      NCHW
    Model
      Convolution
      Activation
      Pooling
    Training
      Forward
      Loss
      Backward
      Optimizer
    Validation
      eval mode
      no_grad
      Metrics
    QC
      Curves
      Predictions
      Tiny overfit

```


## 1. Build the simplest useful baseline first

A baseline answers: **Can a straightforward model learn useful signal at all?**

Before U-Net/transformers/diffusion:
- verify data loader;
- verify shapes;
- train a small CNN;
- confirm loss decreases;
- run tiny-set overfit;
- inspect predictions.

A complicated model makes debugging harder because more components can fail.


## 2. Convolution conceptually

A 2-D convolution applies learned local filters across an image. For one output feature map:

\[
Y_{i,j}=\sum_{c,u,v}K_{c,u,v}X_{c,i+u,j+v}+b
\]

The network learns kernels \(K\) rather than using hand-designed filters.

Early layers often respond to local edges/textures; deeper layers combine information over a larger **receptive field**.


In [ ]:
import torch
import torch.nn as nn

conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)
x = torch.randn(4, 1, 128, 128)
y = conv(x)

print("input :", x.shape)
print("output:", y.shape)


## 3. A small classifier baseline

This example is intentionally simple. It teaches module structure, forward pass, and shape reasoning.


In [ ]:
import torch
import torch.nn as nn

class SmallCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

model = SmallCNN()
x = torch.randn(8, 1, 128, 128)
print(model(x).shape)   # [batch, classes]


## 4. One correct training step

The order matters:

```text
model.train()
zero gradients
forward
loss
backward
optimizer.step()
```

`optimizer.zero_grad()` is needed because PyTorch gradients accumulate by default.


In [ ]:
import torch
import torch.nn as nn

model = SmallCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

images = torch.randn(8, 1, 128, 128)
labels = torch.randint(0, 2, (8,))

model.train()
optimizer.zero_grad(set_to_none=True)
logits = model(images)
loss = criterion(logits, labels)
loss.backward()
optimizer.step()

print("loss:", loss.item())


## 5. Training loop versus validation loop

Validation differs in three important ways:

```python
model.eval()
with torch.no_grad():
    ...
```

No optimizer step occurs. `model.eval()` also changes behavior of layers such as dropout and batch normalization.


## 6. What to plot every experiment

At minimum:
- training loss vs epoch;
- validation loss vs epoch;
- primary validation metric vs epoch;
- learning rate if scheduled;
- representative predictions;
- failure cases.

A single final metric hides training instability and overfitting.


## 7. Baseline acceptance checklist

Do not proceed to a complex model until:

- [ ] inputs and targets have correct shape/range;
- [ ] training loss decreases;
- [ ] tiny-set overfit works;
- [ ] validation code is independent from training;
- [ ] split is specimen-level;
- [ ] metric implementation has a unit test/simple known case;
- [ ] predictions are visually inspected;
- [ ] random baseline/simple classical baseline is known when meaningful.


## End-of-notebook checklist

Before moving on, you should be able to explain the main ideas **without looking at the code**. If you cannot explain why a method, loss, split, or metric is appropriate, repeat the relevant section before using it in research.
